In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()


Saving AQI station day.xlsx to AQI station day.xlsx
Saving Noise station month.xlsx to Noise station month.xlsx


In [3]:
aqi_df = pd.read_excel("AQI station day.xlsx")
noise_df = pd.read_excel("Noise station month.xlsx")


In [4]:
print(aqi_df.shape)
print(noise_df.shape)


(95719, 19)
(5005, 10)


In [5]:
aqi_df['Date'] = pd.to_datetime(aqi_df['Date'], errors='coerce')

aqi_df['Year'] = aqi_df['Date'].dt.year
aqi_df['Month'] = aqi_df['Date'].dt.month


In [7]:
aqi_cols = [
    'city', 'Year', 'Month',
    'PM2.5', 'PM10', 'NO2', 'SO2', 'CO', 'O3', 'AQI'
]

aqi_df = aqi_df[aqi_cols]


In [9]:
aqi_df = aqi_df.dropna(subset=['city', 'Year', 'Month'])

aqi_df[['PM2.5','PM10','NO2','SO2','CO','O3','AQI']] = (
    aqi_df[['PM2.5','PM10','NO2','SO2','CO','O3','AQI']]
    .fillna(aqi_df.groupby('city')[['PM2.5','PM10','NO2','SO2','CO','O3','AQI']].transform('median'))
)


In [10]:
aqi_monthly = aqi_df.groupby(['city', 'Year', 'Month']).agg(
    avg_PM25=('PM2.5', 'mean'),
    avg_PM10=('PM10', 'mean'),
    avg_NO2=('NO2', 'mean'),
    avg_SO2=('SO2', 'mean'),
    avg_CO=('CO', 'mean'),
    avg_O3=('O3', 'mean'),
    avg_AQI=('AQI', 'mean'),
    max_AQI=('AQI', 'max'),
    std_PM25=('PM2.5', 'std')
).reset_index()


In [11]:
aqi_monthly.head()


,city,Year,Month,avg_PM25,avg_PM10,avg_NO2,avg_SO2,avg_CO,avg_O3,avg_AQI,max_AQI,std_PM25
0,Ahmedabad,2015,1,61.507097,107.96,26.846774,43.602903,22.876290,46.350645,381.193548,514.0,8.750000
1,Ahmedabad,2015,2,116.101600,107.96,31.315200,63.194000,21.820000,48.650400,520.640000,1247.0,49.916896
2,Ahmedabad,2015,3,110.469333,107.96,27.937333,58.874333,14.038333,45.752667,416.300000,883.0,44.481601
3,Ahmedabad,2015,4,101.682000,107.96,20.754000,51.233333,7.306333,31.376000,321.283333,774.0,32.912830
4,Ahmedabad,2015,5,74.919355,107.96,17.325806,35.977419,8.529677,31.624194,267.370968,577.0,28.643802


In [12]:
noise_df.columns = noise_df.columns.str.strip()


In [13]:
noise_df = noise_df[['City', 'Year', 'Month', 'Day', 'Night']]


In [14]:
noise_df[['Day','Night']] = noise_df[['Day','Night']].fillna(
    noise_df.groupby('City')[['Day','Night']].transform('median')
)


In [16]:
aqi_monthly.rename(columns={'city': 'City'}, inplace=True)

In [17]:
final_df = pd.merge(
    aqi_monthly,
    noise_df,
    on=['City', 'Year', 'Month'],
    how='inner'
)


In [18]:
print(final_df.shape)
final_df.head()


(2796, 14)


,City,Year,Month,avg_PM25,avg_PM10,avg_NO2,avg_SO2,avg_CO,avg_O3,avg_AQI,max_AQI,std_PM25,Day,Night
0,Bengaluru,2015,1,30.07,73.44,19.537419,23.758871,9.05,26.06871,83.0,83.0,0.0,59.0,56.0
1,Bengaluru,2015,1,30.07,73.44,19.537419,23.758871,9.05,26.06871,83.0,83.0,0.0,66.0,58.0
2,Bengaluru,2015,1,30.07,73.44,19.537419,23.758871,9.05,26.06871,83.0,83.0,0.0,54.0,50.0
3,Bengaluru,2015,1,30.07,73.44,19.537419,23.758871,9.05,26.06871,83.0,83.0,0.0,65.0,56.0
4,Bengaluru,2015,1,30.07,73.44,19.537419,23.758871,9.05,26.06871,83.0,83.0,0.0,58.0,56.0


In [19]:
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Summer'
    elif month in [6, 7, 8, 9]:
        return 'Monsoon'
    else:
        return 'Post-Monsoon'

final_df['Season'] = final_df['Month'].apply(get_season)


In [20]:
final_df = pd.get_dummies(final_df, columns=['Season'], drop_first=True)


In [21]:
air_features = [
    'Year', 'Month',
    'avg_PM25', 'avg_PM10', 'avg_NO2', 'avg_SO2', 'avg_CO', 'avg_O3'
]

air_target = 'avg_AQI'

air_model_df = final_df[['City'] + air_features + [air_target]]


In [22]:
noise_features = [
    'Year', 'Month',
    'avg_PM25', 'avg_PM10', 'avg_NO2', 'avg_AQI'
]

noise_targets = ['Day', 'Night']

noise_model_df = final_df[['City'] + noise_features + noise_targets]


In [23]:
air_model_df.to_csv("air_pollution_model_data.csv", index=False)
noise_model_df.to_csv("noise_pollution_model_data.csv", index=False)
final_df.to_csv("combined_air_noise_data.csv", index=False)


In [24]:
files.download("air_pollution_model_data.csv")
files.download("noise_pollution_model_data.csv")
files.download("combined_air_noise_data.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>